In [40]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime

In [41]:
import pandas as pd

df = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block1_train.parquet')
df.head()

,quote_id,vehicle_ownership_duration,vehicle_number_plate,coverage,claim_free_years,payment_frequency,contractor_birthdate,is_driver_owner,usage,second_driver_birthdate,...,Insurer_B_price,Insurer_C_price,Insurer_D_price,Insurer_E_price,Insurer_F_price,Insurer_G_price,Insurer_H_price,Insurer_I_price,Insurer_J_price,Insurer_K_price
0,1,0.0,b37e7a3f19,mtpl,2,monthly,09/08/2005,yes,private_use,None,...,NaN,NaN,NaN,NaN,101.10,NaN,NaN,NaN,NaN,NaN
1,2,0.0,9d65089849,limited_casco,0,monthly,06/02/2002,yes,private_use,None,...,NaN,258.93,362.20,298.19,280.10,NaN,NaN,NaN,331.69,607.38
2,3,1.0,ec2c02194e,mtpl,5,monthly,31/05/2000,yes,private_use,None,...,82.39,NaN,59.47,122.63,NaN,59.23,66.09,NaN,NaN,73.74
3,4,0.0,9d65089849,limited_casco,5,monthly,06/02/2002,yes,private_use,None,...,NaN,174.81,246.30,211.29,191.73,NaN,NaN,NaN,199.26,370.03
4,5,0.0,27c0f9d91d,limited_casco,3,monthly,21/09/2005,yes,private_use,None,...,202.14,NaN,NaN,NaN,141.07,NaN,NaN,214.38,NaN,163.05


In [42]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()

(541292, 155)
quote_id                        int64
vehicle_ownership_duration     object
vehicle_number_plate           object
coverage                       object
claim_free_years               object
                               ...   
Insurer_G_price               float64
Insurer_H_price               float64
Insurer_I_price               float64
Insurer_J_price               float64
Insurer_K_price               float64
Length: 155, dtype: object
quote_id                           0
vehicle_ownership_duration     22575
vehicle_number_plate               0
coverage                           0
claim_free_years                   0
                               ...  
Insurer_G_price               227071
Insurer_H_price               427889
Insurer_I_price               251863
Insurer_J_price               235193
Insurer_K_price               183605
Length: 155, dtype: int64


,quote_id,vehicle_ownership_duration,vehicle_number_plate,coverage,claim_free_years,payment_frequency,contractor_birthdate,is_driver_owner,usage,second_driver_birthdate,...,Insurer_B_price,Insurer_C_price,Insurer_D_price,Insurer_E_price,Insurer_F_price,Insurer_G_price,Insurer_H_price,Insurer_I_price,Insurer_J_price,Insurer_K_price
0,1,0.0,b37e7a3f19,mtpl,2,monthly,09/08/2005,yes,private_use,None,...,NaN,NaN,NaN,NaN,101.10,NaN,NaN,NaN,NaN,NaN
1,2,0.0,9d65089849,limited_casco,0,monthly,06/02/2002,yes,private_use,None,...,NaN,258.93,362.20,298.19,280.10,NaN,NaN,NaN,331.69,607.38
2,3,1.0,ec2c02194e,mtpl,5,monthly,31/05/2000,yes,private_use,None,...,82.39,NaN,59.47,122.63,NaN,59.23,66.09,NaN,NaN,73.74
3,4,0.0,9d65089849,limited_casco,5,monthly,06/02/2002,yes,private_use,None,...,NaN,174.81,246.30,211.29,191.73,NaN,NaN,NaN,199.26,370.03
4,5,0.0,27c0f9d91d,limited_casco,3,monthly,21/09/2005,yes,private_use,None,...,202.14,NaN,NaN,NaN,141.07,NaN,NaN,214.38,NaN,163.05


In [43]:
reference_date = pd.Timestamp.today()

# Convert birthdates to ages
df['contractor_age'] = (reference_date - pd.to_datetime(df['contractor_birthdate'], format="%d/%m/%Y")).dt.days // 365
df['second_driver_age'] = (reference_date - pd.to_datetime(df['second_driver_birthdate'], format="%d/%m/%Y")).dt.days // 365

# Drop original date columns
df.drop(columns=['contractor_birthdate', 'second_driver_birthdate'], inplace=True)

In [44]:
# vehicle_number_plate is an ID-like column, not useful for prediction
df.drop(columns=['quote_id', 'vehicle_number_plate'], inplace=True)

In [45]:
print(df.columns.tolist())

['vehicle_ownership_duration', 'coverage', 'claim_free_years', 'payment_frequency', 'is_driver_owner', 'usage', 'second_driver_claim_free_years', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_engine_size', 'vehicle_power', 'vehicle_net_weight', 'vehicle_gross_weight', 'vehicle_length', 'vehicle_width', 'vehicle_height', 'vehicle_number_of_cylinders', 'vehicle_number_of_doors', 'vehicle_number_of_seats', 'vehicle_number_of_wheels', 'vehicle_primary_color', 'vehicle_value_new', 'vehicle_net_max_power', 'vehicle_net_max_power_electric', 'vehicle_nominal_continuous_max_power', 'vehicle_power_to_net_weight_ratio', 'vehicle_age', 'vehicle_first_registration_date', 'vehicle_country_first_registration_date', 'vehicle_last_registration_date', 'vehicle_years_since_country_first_registration', 'vehicle_inspection_report_date', 'vehicle_inspection_expiry_date', 'vehicle_inspection_number_of_deficiencies_found', 'vehicle_year_of_last_odometer_report', 'vehicle_odometer_verdict_cod

In [46]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

In [47]:
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['Insurer_A_deductible', 'Insurer_B_deductible', 'Insurer_C_deductible', 'Insurer_D_deductible', 'Insurer_E_deductible', 'Insurer_F_deductible', 'Insurer_G_deductible', 'Insurer_H_deductible', 'Insurer_I_deductible', 'Insurer_J_deductible', 'Insurer_K_deductible', 'Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price', 'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price', 'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price', 'Insurer_J_price', 'Insurer_K_price', 'contractor_age', 'second_driver_age']
Categorical columns: ['vehicle_ownership_duration', 'coverage', 'claim_free_years', 'payment_frequency', 'is_driver_owner', 'usage', 'second_driver_claim_free_years', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_engine_size', 'vehicle_power', 'vehicle_net_weight', 'vehicle_gross_weight', 'vehicle_length', 'vehicle_width', 'vehicle_height', 'vehicle_number_of_cylinders', 'vehicle_number_of_doors', 'vehicle_number_of_seats', 'vehicle_number_of_whe

In [48]:
# Full null summary
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct,
    'dtype': df.dtypes
}).query('null_count > 0').sort_values('null_pct', ascending=False)

print(f"Total columns: {df.shape[1]}")
print(f"Columns WITH nulls: {len(null_summary)}")
print(f"Columns WITHOUT nulls: {df.shape[1] - len(null_summary)}")
print()
print(null_summary.to_string())

Total columns: 153
Columns WITH nulls: 134
Columns WITHOUT nulls: 19

                                                            null_count  null_pct    dtype
second_driver_age                                               541292    100.00  float64
postal_code_houses_owned_by_rental_association_ratio            541292    100.00   object
second_driver_claim_free_years                                  541292    100.00   object
postal_code_houses_built_05_to_15_ratio                         483472     89.32   object
postal_code_houses_built_95_to_05_ratio                         474873     87.73   object
postal_code_houses_built_85_to_95_ratio                         465567     86.01   object
postal_code_houses_built_75_to_85_ratio                         457575     84.53   object
postal_code_houses_built_45_to_65_ratio                         450976     83.31   object
postal_code_houses_built_65_to_75_ratio                         446859     82.55   object
postal_code_houses_built_befor

In [49]:
high_null   = null_summary[null_summary['null_pct'] > 50].index.tolist()
medium_null = null_summary[(null_summary['null_pct'] > 5) & 
                           (null_summary['null_pct'] <= 50)].index.tolist()
low_null    = null_summary[null_summary['null_pct'] <= 5].index.tolist()

print(f"High   (>50% missing) — {len(high_null)} cols:   consider dropping")
print(f"Medium (5–50% missing) — {len(medium_null)} cols: impute carefully")
print(f"Low    (<5% missing)  — {len(low_null)} cols:   safe to impute")

High   (>50% missing) — 18 cols:   consider dropping
Medium (5–50% missing) — 53 cols: impute carefully
Low    (<5% missing)  — 63 cols:   safe to impute


In [50]:
# Drop 100% null columns
cols_100_null = [col for col in df.columns if df[col].isnull().mean() == 1.0]
print(f"Dropping {len(cols_100_null)} fully null columns: {cols_100_null}")
df.drop(columns=cols_100_null, inplace=True)

Dropping 3 fully null columns: ['second_driver_claim_free_years', 'postal_code_houses_owned_by_rental_association_ratio', 'second_driver_age']


In [51]:
# All columns that are clearly numeric but stored as object
to_numeric_cols = [
    # Vehicle physical specs
    'vehicle_engine_size',
    'vehicle_power',
    'vehicle_net_weight',
    'vehicle_gross_weight',
    'vehicle_length',
    'vehicle_width',
    'vehicle_height',
    'vehicle_net_max_power',
    'vehicle_net_max_power_electric',
    'vehicle_nominal_continuous_max_power',
    'vehicle_power_to_net_weight_ratio',
    'vehicle_number_of_cylinders',
    'vehicle_number_of_doors',
    'vehicle_number_of_seats',
    'vehicle_number_of_wheels',
    'vehicle_value_new',
    'vehicle_planned_annual_mileage',
    'vehicle_age',
    'vehicle_years_since_country_first_registration',
    'vehicle_inspection_number_of_deficiencies_found',
    'vehicle_year_of_last_odometer_report',

    # Location numeric
    'postal_code_latitude',
    'postal_code_longitude',
    'postal_code_distance_to_border',
    'postal_code_population',
    'postal_code_households',
    'postal_code_houses',
    'postal_code_average_household_size',
    'postal_code_average_property_value',
    'postal_code_address_density',
    'municipality_crimes_per_1000',

    # Postal code age ratios
    'postal_code_0_to_15_years_inhabitants_ratio',
    'postal_code_25_to_45_years_inhabitants_ratio',
    'postal_code_45_to_65_years_inhabitants_ratio',
    'postal_code_65_years_and_older_inhabitants_ratio',
    'postal_code_single_person_households_ratio',
    'postal_code_multi_person_households_without_children_ratio',
    'postal_code_two_parent_households_ratio',
    'postal_code_social_benefit_recipients_ratio',
    'postal_code_owner_occupied_houses_ratio',
    'postal_code_rental_houses_ratio',
    'postal_code_multi_family_houses_ratio',

    # House build era ratios
    'postal_code_houses_built_before_1945_ratio',
    'postal_code_houses_built_45_to_65_ratio',
    'postal_code_houses_built_65_to_75_ratio',
    'postal_code_houses_built_75_to_85_ratio',
    'postal_code_houses_built_85_to_95_ratio',
    'postal_code_houses_built_95_to_05_ratio',
    'postal_code_houses_built_05_to_15_ratio',

    # Distance to nearest amenity
    'postal_code_nearest_train_station_distance',
    'postal_code_nearest_train_transfer_station_distance',
    'postal_code_nearest_motorway_ramp_distance',
    'postal_code_nearest_hospital_excl_clinic_distance',
    'postal_code_nearest_hospital_incl_clinic_distance',
    'postal_code_nearest_general_practitioner_distance',
    'postal_code_nearest_gp_post_service_distance',
    'postal_code_nearest_pharmacy_distance',
    'postal_code_nearest_primary_school_distance',
    'postal_code_nearest_secondary_school_distance',
    'postal_code_nearest_secondary_school_havo_vwo_distance',
    'postal_code_nearest_secondary_school_vmbo_distance',
    'postal_code_nearest_supermarket_distance',
    'postal_code_nearest_groceries_distance',
    'postal_code_nearest_restaurant_distance',
    'postal_code_nearest_cafe_distance',
    'postal_code_nearest_snackbar_distance',
    'postal_code_nearest_hotel_distance',
    'postal_code_nearest_library_distance',
    'postal_code_nearest_museum_distance',
    'postal_code_nearest_cinema_distance',
    'postal_code_nearest_department_store_distance',
    'postal_code_nearest_fire_station_distance',
    'postal_code_nearest_daycare_distance',
    'postal_code_nearest_after_school_care_distance',
    'postal_code_nearest_attraction_distance',
    'postal_code_nearest_performance_venue_distance',
    'postal_code_nearest_music_venue_distance',
    'postal_code_nearest_swimming_pool_distance',
    'postal_code_nearest_ice_rink_distance',
    'postal_code_nearest_sauna_distance',
    'postal_code_nearest_tanning_salon_distance',

    # Count of amenities within radius
    'postal_code_hospitals_excl_clinic_within_10_km',
    'postal_code_hospitals_incl_clinic_within_10_km',
    'postal_code_general_practitioners_within_3_km',
    'postal_code_primary_schools_within_3_km',
    'postal_code_secondary_schools_within_5_km',
    'postal_code_secondary_schools_havo_vwo_within_5_km',
    'postal_code_secondary_schools_vmbo_within_5_km',
    'postal_code_supermarkets_within_3_km',
    'postal_code_groceries_within_3_km',
    'postal_code_restaurants_within_3_km',
    'postal_code_cafes_within_3_km',
    'postal_code_snackbars_within_3_km',
    'postal_code_hotels_within_10_km',
    'postal_code_museums_within_10_km',
    'postal_code_cinemas_within_10_km',
    'postal_code_department_stores_within_10_km',
    'postal_code_daycares_within_3_km',
    'postal_code_after_school_cares_within_3_km',
    'postal_code_attractions_within_20_km',
    'postal_code_performance_venues_within_10_km',
]

# Only convert columns that still exist after dropping 100% null ones
to_numeric_cols = [c for c in to_numeric_cols if c in df.columns]

for col in to_numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Verify results
converted = df[to_numeric_cols].dtypes
still_object = converted[converted == 'object']

print(f"Successfully converted: {len(to_numeric_cols) - len(still_object)} columns")
print(f"Still object after conversion: {len(still_object)}")
if len(still_object) > 0:
    print(still_object)

Successfully converted: 101 columns
Still object after conversion: 0


In [52]:
df = df.copy()

reference = pd.Timestamp.today()

date_cols = [
    'vehicle_first_registration_date',
    'vehicle_country_first_registration_date',
    'vehicle_last_registration_date',
    'vehicle_inspection_report_date',
    'vehicle_inspection_expiry_date',
]

# Only work with date cols that still exist
date_cols = [c for c in date_cols if c in df.columns]
print(f"Date cols still present: {date_cols}")

# Flag inspection report missingness (only if col still exists)
if 'vehicle_inspection_report_date' in date_cols:
    df['has_inspection_report'] = df['vehicle_inspection_report_date'].notnull().astype(int)

# Parse dates with correct format
for col in date_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Create derived features only if source col exists and derived col doesn't yet
if 'vehicle_first_registration_date' in date_cols and \
   'vehicle_years_since_first_registration' not in df.columns:
    df['vehicle_years_since_first_registration'] = (
        (reference - df['vehicle_first_registration_date']).dt.days // 365
    )

if 'vehicle_inspection_expiry_date' in date_cols:
    df['vehicle_days_until_inspection_expiry'] = (
        df['vehicle_inspection_expiry_date'] - reference
    ).dt.days

if 'vehicle_last_registration_date' in date_cols and \
   'vehicle_years_since_last_registration' not in df.columns:
    df['vehicle_years_since_last_registration'] = (
        (reference - df['vehicle_last_registration_date']).dt.days // 365
    )

df.drop(columns=date_cols, inplace=True)
df = df.copy()

print("\nDerived date columns:")
derived = [c for c in [
    'has_inspection_report',
    'vehicle_years_since_first_registration',
    'vehicle_days_until_inspection_expiry',
    'vehicle_years_since_last_registration'
] if c in df.columns]
print(df[derived].describe())

Date cols still present: ['vehicle_first_registration_date', 'vehicle_country_first_registration_date', 'vehicle_last_registration_date', 'vehicle_inspection_report_date', 'vehicle_inspection_expiry_date']

Derived date columns:
       has_inspection_report  vehicle_years_since_first_registration  \
count          541292.000000                           539804.000000   
mean                0.588082                               11.204935   
std                 0.492181                                6.393046   
min                 0.000000                                0.000000   
25%                 0.000000                                6.000000   
50%                 1.000000                               11.000000   
75%                 1.000000                               16.000000   
max                 1.000000                               41.000000   

       vehicle_days_until_inspection_expiry  \
count                         539451.000000   
mean                        

In [53]:
print(f"Shape: {df.shape}")
print(f"\nDtype counts:")
print(df.dtypes.value_counts())
print(f"\nRemaining object columns:")
print(df.select_dtypes(include='object').columns.tolist())

Shape: (541292, 149)

Dtype counts:
float64    123
object      21
int64        5
Name: count, dtype: int64

Remaining object columns:
['vehicle_ownership_duration', 'coverage', 'claim_free_years', 'payment_frequency', 'is_driver_owner', 'usage', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'vehicle_is_imported', 'vehicle_is_imported_within_last_12_months', 'vehicle_can_be_registered', 'vehicle_has_open_recall', 'vehicle_is_marked_for_export', 'vehicle_is_taxi', 'postal_code', 'province', 'municipality', 'postal_code_urban_category']


In [54]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct,
    'dtype': df.dtypes
}).query('null_count > 0').sort_values('null_pct', ascending=False)

print(f"Columns with nulls: {len(null_summary)} / {df.shape[1]}")
print(null_summary.to_string())

Columns with nulls: 129 / 149
                                                            null_count  null_pct    dtype
postal_code_houses_built_05_to_15_ratio                         483472     89.32  float64
postal_code_houses_built_95_to_05_ratio                         474873     87.73  float64
postal_code_houses_built_85_to_95_ratio                         465567     86.01  float64
postal_code_houses_built_75_to_85_ratio                         457575     84.53  float64
postal_code_houses_built_45_to_65_ratio                         450976     83.31  float64
postal_code_houses_built_65_to_75_ratio                         446859     82.55  float64
postal_code_houses_built_before_1945_ratio                      434303     80.23  float64
Insurer_H_deductible                                            427889     79.05  float64
Insurer_H_price                                                 427889     79.05  float64
vehicle_net_max_power_electric                                  421726

In [55]:
df.select_dtypes(include='number').describe().T.sort_values('mean', ascending=False)

,count,mean,std,min,25%,50%,75%,max
vehicle_value_new,538563.0,35079.178705,22715.015016,500.000000,18999.000000,29455.000000,44984.000000,1579134.00
vehicle_planned_annual_mileage,541292.0,13912.127502,6327.402285,7500.000000,10000.000000,12000.000000,20000.000000,30001.00
vehicle_year_of_last_odometer_report,539802.0,2024.901106,0.418362,1992.000000,2025.000000,2025.000000,2025.000000,2025.00
vehicle_gross_weight,541197.0,1813.849290,391.452234,820.000000,1550.000000,1785.000000,2030.000000,3500.00
vehicle_engine_size,505005.0,1504.412772,510.140289,462.000000,1197.000000,1395.000000,1796.000000,6716.00
...,...,...,...,...,...,...,...,...
postal_code_45_to_65_years_inhabitants_ratio,472013.0,0.285134,0.107013,0.007092,0.202899,0.285714,0.350000,1.00
postal_code_65_years_and_older_inhabitants_ratio,327934.0,0.224658,0.151452,0.005556,0.125000,0.187500,0.285714,1.00
postal_code_0_to_15_years_inhabitants_ratio,358038.0,0.198688,0.083499,0.005917,0.136364,0.190476,0.250000,0.80
postal_code_social_benefit_recipients_ratio,216049.0,0.163545,0.103936,0.006231,0.084507,0.142857,0.222222,1.00


In [56]:
# Drop house build era ratio columns — 80-89% null, not worth imputing
house_build_cols = [col for col in df.columns if 'houses_built' in col]
print(f"Dropping {len(house_build_cols)} columns:")
print(house_build_cols)

df.drop(columns=house_build_cols, inplace=True)
print(f"\nNew shape: {df.shape}")

Dropping 7 columns:
['postal_code_houses_built_before_1945_ratio', 'postal_code_houses_built_45_to_65_ratio', 'postal_code_houses_built_65_to_75_ratio', 'postal_code_houses_built_75_to_85_ratio', 'postal_code_houses_built_85_to_95_ratio', 'postal_code_houses_built_95_to_05_ratio', 'postal_code_houses_built_05_to_15_ratio']

New shape: (541292, 142)


In [57]:
# Add binary 'quoted' flag for each insurer, 1 = they provided a quote, 0 = they didn't
insurer_price_cols = [col for col in df.columns if col.endswith('_price') and col.startswith('Insurer')]

for col in insurer_price_cols:
    insurer_name = col.replace('_price', '')
    df[f'{insurer_name}_quoted'] = df[col].notnull().astype(int)

# Verify
quoted_cols = [col for col in df.columns if col.endswith('_quoted')]
print(f"Added {len(quoted_cols)} quoted flag columns:")
print(df[quoted_cols].sum().sort_values(ascending=False).to_frame('quotes_count'))

Added 11 quoted flag columns:
                  quotes_count
Insurer_A_quoted        527372
Insurer_B_quoted        435759
Insurer_C_quoted        431011
Insurer_D_quoted        403079
Insurer_E_quoted        402556
Insurer_F_quoted        357844
Insurer_K_quoted        357687
Insurer_G_quoted        314221
Insurer_J_quoted        306099
Insurer_I_quoted        289429
Insurer_H_quoted        113403


In [58]:
# Null = not electric, fill with 0
electric_cols = [
    'vehicle_net_max_power_electric',
    'vehicle_nominal_continuous_max_power'
]

# Add a flag first to capture "is electric" signal
df['vehicle_is_electric'] = df['vehicle_net_max_power_electric'].notnull().astype(int)

df[electric_cols] = df[electric_cols].fillna(0)

print(f"Electric vehicles in dataset: {df['vehicle_is_electric'].sum()} ({df['vehicle_is_electric'].mean()*100:.2f}%)")
print(df[electric_cols + ['vehicle_is_electric']].describe())

Electric vehicles in dataset: 119566 (22.09%)
       vehicle_net_max_power_electric  vehicle_nominal_continuous_max_power  \
count                   541292.000000                         541292.000000   
mean                        17.394362                              9.704313   
std                         45.356583                             23.644732   
min                          0.000000                              0.000000   
25%                          0.000000                              0.000000   
50%                          0.000000                              0.000000   
75%                          0.000000                              0.000000   
max                        386.000000                            250.000000   

       vehicle_is_electric  
count        541292.000000  
mean              0.220890  
std               0.414847  
min               0.000000  
25%               0.000000  
50%               0.000000  
75%               0.000000  
max       

In [59]:
# These are all yes/no or true/false columns stored as strings
binary_cols = [
    'vehicle_is_imported',
    'vehicle_is_imported_within_last_12_months',
    'vehicle_can_be_registered',
    'vehicle_has_open_recall',
    'vehicle_is_marked_for_export',
    'vehicle_is_taxi',
    'is_driver_owner',
]

# First peek at unique values to know what we're mapping
for col in binary_cols:
    print(f"{col}: {df[col].unique()}")

vehicle_is_imported: ['yes' 'no']
vehicle_is_imported_within_last_12_months: ['no' 'yes' None]
vehicle_can_be_registered: ['yes' 'no']
vehicle_has_open_recall: ['no' 'yes']
vehicle_is_marked_for_export: ['no' 'yes']
vehicle_is_taxi: ['no']
is_driver_owner: ['yes' 'no']


In [60]:
# Safe version — won't error if already dropped
if 'vehicle_is_taxi' in df.columns:
    df.drop(columns=['vehicle_is_taxi'], inplace=True)
    print("Dropped vehicle_is_taxi")
else:
    print("vehicle_is_taxi already dropped, skipping")

# Map yes/no to 1/0 — safe to rerun, already mapped values won't be in the mapping
binary_cols = [
    'vehicle_is_imported',
    'vehicle_is_imported_within_last_12_months',
    'vehicle_can_be_registered',
    'vehicle_has_open_recall',
    'vehicle_is_marked_for_export',
    'is_driver_owner',
]

mapping = {'yes': 1, 'no': 0}
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map(mapping)

print(f"\nNew shape: {df.shape}")
print("\nNull counts after mapping:")
print(df[binary_cols].isnull().sum())
print("\nValue counts sample:")
print(df[binary_cols].sum().to_frame('count_of_1s'))

Dropped vehicle_is_taxi

New shape: (541292, 153)

Null counts after mapping:
vehicle_is_imported                              0
vehicle_is_imported_within_last_12_months    21451
vehicle_can_be_registered                        0
vehicle_has_open_recall                          0
vehicle_is_marked_for_export                     0
is_driver_owner                                  0
dtype: int64

Value counts sample:
                                           count_of_1s
vehicle_is_imported                           157696.0
vehicle_is_imported_within_last_12_months      31956.0
vehicle_can_be_registered                     538363.0
vehicle_has_open_recall                        42112.0
vehicle_is_marked_for_export                    1398.0
is_driver_owner                               540834.0


In [61]:
print(df[binary_cols].isnull().sum())
print(f"\nShape: {df.shape}")
print(f"Remaining object cols: {df.select_dtypes(include='object').columns.tolist()}")

vehicle_is_imported                              0
vehicle_is_imported_within_last_12_months    21451
vehicle_can_be_registered                        0
vehicle_has_open_recall                          0
vehicle_is_marked_for_export                     0
is_driver_owner                                  0
dtype: int64

Shape: (541292, 153)
Remaining object cols: ['vehicle_ownership_duration', 'coverage', 'claim_free_years', 'payment_frequency', 'usage', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'postal_code', 'province', 'municipality', 'postal_code_urban_category']


In [62]:
df['vehicle_is_imported_within_last_12_months'] = \
    df['vehicle_is_imported_within_last_12_months'].fillna(0).astype(int)

print(df['vehicle_is_imported_within_last_12_months'].isnull().sum())  # should be 0
print(df['vehicle_is_imported_within_last_12_months'].value_counts())

0
vehicle_is_imported_within_last_12_months
0    509336
1     31956
Name: count, dtype: int64


In [63]:
for col in df.select_dtypes(include='object').columns:
    n_unique = df[col].nunique()
    top3 = df[col].value_counts().head(3).to_dict()
    print(f"{col:45s} unique: {n_unique:6d}  top3: {top3}")

vehicle_ownership_duration                    unique:     33  top3: {'0.0': 259275, '1.0': 80028, '2.0': 49652}
coverage                                      unique:      3  top3: {'limited_casco': 238176, 'casco': 163742, 'mtpl': 139374}
claim_free_years                              unique:     76  top3: {'0': 159429, '1': 50057, '2': 41939}
payment_frequency                             unique:      2  top3: {'monthly': 529806, 'yearly': 11486}
usage                                         unique:      1  top3: {'private_use': 541292}
vehicle_maker                                 unique:    119  top3: {'volkswagen': 78101, 'bmw': 39170, 'peugeot': 34758}
vehicle_model                                 unique:   1055  top3: {'toyota': 32631, 'golf': 27360, 'polo': 24073}
vehicle_fuel_type                             unique:      7  top3: {'petrol': 394097, 'petrol_electric_hybrid': 80616, 'electric': 35453}
vehicle_primary_color                         unique:     15  top3: {'grey': 1836

In [64]:
# Drop zero variance
df.drop(columns=['usage'], inplace=True)
print("Dropped usage")

# Cast numeric strings to numbers
df['vehicle_ownership_duration'] = pd.to_numeric(df['vehicle_ownership_duration'], errors='coerce')
df['claim_free_years'] = pd.to_numeric(df['claim_free_years'], errors='coerce')
df['postal_code_urban_category'] = pd.to_numeric(df['postal_code_urban_category'], errors='coerce')
df['postal_code'] = pd.to_numeric(df['postal_code'], errors='coerce')
print("Cast 4 numeric-string columns to numbers")

# Binary encode payment_frequency
df['payment_frequency'] = df['payment_frequency'].map({'monthly': 0, 'yearly': 1})
print("Encoded payment_frequency")

print(f"\nShape: {df.shape}")
print(f"Remaining object cols: {df.select_dtypes(include='object').columns.tolist()}")

Dropped usage
Cast 4 numeric-string columns to numbers
Encoded payment_frequency

Shape: (541292, 152)
Remaining object cols: ['coverage', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'province', 'municipality']


In [65]:
print("Coverage distribution:")
print(df['coverage'].value_counts())

print("\nCoverage + top insurer quote rates:")
insurer_quoted = [c for c in df.columns if c.endswith('_quoted')]
print(df.groupby('coverage')[insurer_quoted].mean().round(2))

Coverage distribution:
coverage
limited_casco    238176
casco            163742
mtpl             139374
Name: count, dtype: int64

Coverage + top insurer quote rates:
               Insurer_A_quoted  Insurer_B_quoted  Insurer_C_quoted  \
coverage                                                              
casco                      0.96              0.86              0.86   
limited_casco              0.97              0.78              0.81   
mtpl                       1.00              0.78              0.70   

               Insurer_D_quoted  Insurer_E_quoted  Insurer_F_quoted  \
coverage                                                              
casco                      0.82              0.63              0.77   
limited_casco              0.67              0.74              0.62   
mtpl                       0.79              0.89              0.60   

               Insurer_G_quoted  Insurer_H_quoted  Insurer_I_quoted  \
coverage                                          

In [66]:
# Columns that define a unique person/vehicle (everything except price, deductible, quoted flags and coverage)
exclude_cols = (
    [c for c in df.columns if c.endswith('_price')] +
    [c for c in df.columns if c.endswith('_deductible')] +
    [c for c in df.columns if c.endswith('_quoted')] +
    ['coverage']
)

profile_cols = [c for c in df.columns if c not in exclude_cols]

# Find duplicate profiles across different coverages
duplicates = df.groupby(profile_cols)['coverage'].nunique()
multi_coverage = duplicates[duplicates > 1]

print(f"Unique person/vehicle profiles: {len(duplicates)}")
print(f"Profiles appearing in more than one coverage: {len(multi_coverage)}")
print(f"Total rows those profiles represent: {multi_coverage.sum()}")
print(f"\nCoverage combination breakdown:")
print(df.groupby(profile_cols)['coverage'].agg(lambda x: '+'.join(sorted(x.unique()))).value_counts().head(10))

Unique person/vehicle profiles: 1315
Profiles appearing in more than one coverage: 1
Total rows those profiles represent: 2

Coverage combination breakdown:
coverage
limited_casco         806
casco                 440
mtpl                   68
limited_casco+mtpl      1
Name: count, dtype: int64


In [67]:
# For each insurer, use all OTHER insurer prices as features
# Example for Insurer A:
# features = [...profile cols..., Insurer_B_price, Insurer_C_price, ..., Insurer_K_price]
# target = Insurer_A_price

# Quick correlation check to see if this relationship exists
insurer_price_cols = [c for c in df.columns if c.endswith('_price') 
                      and c.startswith('Insurer')]

print("Insurer price correlations:")
print(df[insurer_price_cols].corr().round(2))

Insurer price correlations:
                 Insurer_A_price  Insurer_B_price  Insurer_C_price  \
Insurer_A_price             1.00             0.92             0.94   
Insurer_B_price             0.92             1.00             0.89   
Insurer_C_price             0.94             0.89             1.00   
Insurer_D_price             0.93             0.91             0.91   
Insurer_E_price             0.85             0.84             0.82   
Insurer_F_price             0.90             0.89             0.90   
Insurer_G_price             0.86             0.90             0.85   
Insurer_H_price             0.59             0.53             0.69   
Insurer_I_price             0.77             0.72             0.70   
Insurer_J_price             0.84             0.79             0.82   
Insurer_K_price             0.88             0.88             0.84   

                 Insurer_D_price  Insurer_E_price  Insurer_F_price  \
Insurer_A_price             0.93             0.85            

In [68]:
# Check the actual score-relevant rows per insurer
insurer_price_cols = [c for c in df.columns if c.endswith('_price') and c.startswith('Insurer')]

print("Rows that actually count toward your score:")
total_scored_rows = 0
for col in insurer_price_cols:
    n = df[col].notnull().sum()
    total_scored_rows += n
    print(f"  {col:25s} {n:7,d} rows")

print(f"\nTotal scored predictions: {total_scored_rows:,}")
print(f"Average per insurer:      {total_scored_rows//11:,}")

Rows that actually count toward your score:
  Insurer_A_price           527,372 rows
  Insurer_B_price           435,759 rows
  Insurer_C_price           431,011 rows
  Insurer_D_price           403,079 rows
  Insurer_E_price           402,556 rows
  Insurer_F_price           357,844 rows
  Insurer_G_price           314,221 rows
  Insurer_H_price           113,403 rows
  Insurer_I_price           289,429 rows
  Insurer_J_price           306,099 rows
  Insurer_K_price           357,687 rows

Total scored predictions: 3,938,460
Average per insurer:      358,041


In [69]:
deductible_cols = [c for c in df.columns if c.endswith('_deductible')]
print("Deductible null rates:")
for col in deductible_cols:
    pct = df[col].isnull().mean() * 100
    print(f"  {col:30s} {pct:.1f}% missing")

Deductible null rates:
  Insurer_A_deductible           2.6% missing
  Insurer_B_deductible           19.5% missing
  Insurer_C_deductible           20.4% missing
  Insurer_D_deductible           25.5% missing
  Insurer_E_deductible           25.6% missing
  Insurer_F_deductible           33.9% missing
  Insurer_G_deductible           41.9% missing
  Insurer_H_deductible           79.0% missing
  Insurer_I_deductible           46.5% missing
  Insurer_J_deductible           43.5% missing
  Insurer_K_deductible           33.9% missing


In [70]:
# Check all remaining columns for usefulness
print("=== Potential drop candidates ===\n")

# 1. ID-like columns
print("--- ID / code columns ---")
id_like = ['postal_code', 'vehicle_odometer_verdict_code']
for col in id_like:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")

# 2. Near zero variance numerics
print("\n--- Near zero variance (>99% same value) ---")
for col in df.select_dtypes(include='number').columns:
    top_val_pct = df[col].value_counts(normalize=True).iloc[0]
    if top_val_pct > 0.99:
        top_val = df[col].value_counts().index[0]
        print(f"{col}: {top_val_pct*100:.1f}% = {top_val}")

# 3. High cardinality object columns
print("\n--- High cardinality object columns ---")
for col in df.select_dtypes(include='object').columns:
    print(f"{col}: {df[col].nunique()} unique values")

=== Potential drop candidates ===

--- ID / code columns ---
postal_code: 90 unique values
vehicle_odometer_verdict_code: 8 unique values

--- Near zero variance (>99% same value) ---
is_driver_owner: 99.9% = 1
vehicle_number_of_wheels: 100.0% = 4
vehicle_inspection_number_of_deficiencies_found: 100.0% = 1.0
vehicle_can_be_registered: 99.5% = 1
vehicle_is_marked_for_export: 99.7% = 0
Insurer_D_deductible: 99.8% = 0.0
Insurer_F_deductible: 99.1% = 0.0
Insurer_I_deductible: 99.3% = 0.0

--- High cardinality object columns ---
coverage: 3 unique values
vehicle_maker: 119 unique values
vehicle_model: 1055 unique values
vehicle_fuel_type: 7 unique values
vehicle_primary_color: 15 unique values
vehicle_odometer_verdict_code: 8 unique values
province: 12 unique values
municipality: 360 unique values


In [71]:
# Check the suspicious ones more carefully
print("vehicle_inspection_number_of_deficiencies_found:")
print(df['vehicle_inspection_number_of_deficiencies_found'].value_counts().head(10))

print("\nvehicle_odometer_verdict_code:")
print(df['vehicle_odometer_verdict_code'].value_counts())

print("\nvehicle_model sample — is it actually model or something else?")
print(df['vehicle_model'].value_counts().head(10))

print("\nmunicipality sample:")
print(df['municipality'].value_counts().head(5))

vehicle_inspection_number_of_deficiencies_found:
vehicle_inspection_number_of_deficiencies_found
1.0    318324
Name: count, dtype: int64

vehicle_odometer_verdict_code:
vehicle_odometer_verdict_code
00    350359
05    173723
04     10220
07      3784
ng      2247
01       530
02       293
03        95
Name: count, dtype: int64

vehicle_model sample — is it actually model or something else?
vehicle_model
toyota    32631
golf      27360
polo      24073
fiat      12490
corsa     12163
fiesta    10965
nissan    10006
model      9403
focus      9170
clio       9026
Name: count, dtype: int64

municipality sample:
municipality
Municipality_257    25528
Municipality_18     22765
Municipality_1      19352
Municipality_299    14589
Municipality_11      9637
Name: count, dtype: int64


In [72]:
# Check if this interpretation makes sense
print("has_inspection_report vs deficiencies crosstab:")
print(pd.crosstab(
    df['has_inspection_report'], 
    df['vehicle_inspection_number_of_deficiencies_found'].isnull(),
    rownames=['has_inspection'],
    colnames=['deficiency_is_null']
))

has_inspection_report vs deficiencies crosstab:
deficiency_is_null   False   True 
has_inspection                    
0                        0  222968
1                   318324       0


In [73]:
# Create clean 3-category inspection_status column
df['inspection_status'] = 0  # no inspection
df.loc[(df['has_inspection_report'] == 1) & 
       (df['vehicle_inspection_number_of_deficiencies_found'].isnull()), 
       'inspection_status'] = 1  # inspected, clean
df.loc[(df['has_inspection_report'] == 1) & 
       (df['vehicle_inspection_number_of_deficiencies_found'] == 1.0), 
       'inspection_status'] = 2  # inspected, deficiency found

print(df['inspection_status'].value_counts())

# Now drop the two original columns — all info is captured in inspection_status
df.drop(columns=['has_inspection_report', 
                 'vehicle_inspection_number_of_deficiencies_found'], inplace=True)

print(f"\nNew shape: {df.shape}")

inspection_status
2    318324
0    222968
Name: count, dtype: int64

New shape: (541292, 151)


In [74]:
# Recode to clean binary — 0 = no inspection, 1 = inspected (deficiency found)
df['inspection_status'] = df['inspection_status'].replace({2: 1})

print(df['inspection_status'].value_counts())

inspection_status
1    318324
0    222968
Name: count, dtype: int64


In [75]:
cols_to_drop = [
    'postal_code',
    'vehicle_number_of_wheels',
    'is_driver_owner',
    'vehicle_can_be_registered',
    'vehicle_is_marked_for_export',
    'vehicle_model',
    'municipality',
]

cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f"\nDropped: {cols_to_drop}")
print(f"New shape: {df.shape}")
print(f"Remaining object cols: {df.select_dtypes(include='object').columns.tolist()}")


Dropped: ['postal_code', 'vehicle_number_of_wheels', 'is_driver_owner', 'vehicle_can_be_registered', 'vehicle_is_marked_for_export', 'vehicle_model', 'municipality']
New shape: (541292, 144)
Remaining object cols: ['coverage', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'province']


In [ ]:
# Load test set
df_test = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block2_test.parquet')

print(f"Test shape: {df_test.shape}")
print(f"Train shape: {df.shape}")
print(f"\nTest columns not in train: {set(df_test.columns) - set(df.columns)}")
print(f"Train columns not in test: {set(df.columns) - set(df_test.columns)}")

Test shape: (164092, 144)
Train shape: (541292, 144)

Test columns not in train: {'is_driver_owner', 'vehicle_first_registration_date', 'postal_code_houses_built_85_to_95_ratio', 'vehicle_is_taxi', 'quote_id', 'postal_code_houses_built_45_to_65_ratio', 'postal_code', 'vehicle_inspection_report_date', 'postal_code_houses_built_before_1945_ratio', 'postal_code_houses_built_65_to_75_ratio', 'contractor_birthdate', 'postal_code_houses_built_75_to_85_ratio', 'postal_code_houses_built_95_to_05_ratio', 'second_driver_birthdate', 'usage', 'vehicle_inspection_number_of_deficiencies_found', 'vehicle_number_plate', 'vehicle_inspection_expiry_date', 'vehicle_model', 'postal_code_houses_built_05_to_15_ratio', 'vehicle_country_first_registration_date', 'vehicle_can_be_registered', 'vehicle_number_of_wheels', 'postal_code_houses_owned_by_rental_association_ratio', 'vehicle_is_marked_for_export', 'vehicle_last_registration_date', 'municipality', 'second_driver_claim_free_years'}
Train columns not in t